# Wind data concatenation

This notebook will import SST data from a source, select it to be inside the scope of the project and do EOF analysis to determine Central Atlantic Niño Index and Eastern Atlantic Niño Index.

# Imports

In [3]:
import numpy as np
import xarray as xr
import os
from dask_jobqueue import PBSCluster
from dask.distributed import Client

# Data import

In [18]:
# dates in scope 
hist_decades = ['191001-191912', '192001-192912',
                '193001-193912', '194001-194912', '195001-195912', '196001-196912',
                '197001-197912', '198001-198912', '199001-199912', '200001-200912', 
                '201001-201412']

# too many different member names (macro and micro perturbations)
macro_name = np.char.mod('%d', np.arange(1001, 1311, 10)).tolist()

micro_name = np.char.zfill(np.char.mod('%d', np.arange(1, 21, 1)), 3).tolist()


path = "/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/"
title = "b.e21.BHIST"
type = '.nc'

all = []
for a in ["cmip6", "smbb"]:
    hist = f'{title}{a}'
    for i in macro_name:
        for j in micro_name:
            member = []
            for z in hist_decades:
                file = path + hist + '.f09_g17.LE2-'+ i + '.' +  j + '.cam.h0.U.' + z + type
                # print(file)
                if os.path.exists(file):
                    member.append(file)
                else:
                    continue
            if member != []:
                all.append(member)

In [ ]:
all_ds = []
for m in all:
    member = []
    for f in m:
        print(f)
        ds = xr.open_dataset(f, engine='h5netcdf')['U']
        member.append(ds)
    member_data = xr.concat(member, dim='time')
    all_ds.append(member_data)

/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.191001-191912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.192001-192912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.193001-193912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.194001-194912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.195001-195912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.196001-196912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm/proc/tseries/month_1/U/b.e21.BHISTcmip6.f09_g17.LE2-1001.001.cam.h0.U.197001-197912.nc
/glade/campaign/cgd/cesm/CESM2-LE/timeseries/atm

In [ ]:
ds = xr.concat(all_ds, dim='member')

In [9]:
ds.to_netcdf('/glade/work/acruz/CESM21_Us.nc')